# VanDerPol — Stage 5 — Evaluation & Cross-Method Comparison

Evaluate the configured controllers on matched scenarios.


## 1. Set up the system and load the stage config

Load dependencies and configuration.


In [ ]:
"""Boilerplate: make the in-repo `sdpc` package importable and resolve this system."""
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_p = Path.cwd()
while not (_p / "src" / "sdpc").exists():
    _p = _p.parent
sys.path.insert(0, str(_p / "src"))

import torch
from sdpc.config import load_config
from sdpc.registry import make_system

SYSTEM = "vanderpol_relative_degree_one"
device = torch.device("cpu")
system = make_system(SYSTEM, device=device)
CONFIGS = Path.cwd().parent / "configs"
RESULTS = Path.cwd().parent / "results"

print(f"System        : {SYSTEM}")
print(f"State dim nx  : {system.nx}")
print(f"Control dim nu: {system.nu}")
print(f"Sample time ts: {system.ts}")
print(f"Input bounds  : [{system.umin}, {system.umax}]")
print(f"State bounds  : [{system.xmin}, {system.xmax}]")
print(f"Discrete model: {system.is_discrete}")


In [ ]:
cfg = load_config(CONFIGS / 'eval.yaml')
cfg['n_seeds'] = 5  # reduced for interactive use; increase for the full comparison

print('Methods being compared:', cfg.get('methods'))
print('Number of seeds       :', cfg['n_seeds'])
print('Evaluation steps      :', cfg['eval_nsteps'])
print('SD-DPC/MPC horizon    :', cfg['nsteps'])
print('Shared DPC weights    :', cfg['weights'])

## 2. Load the trained SINDy model and sparse policy

Restore the required checkpoints.


In [ ]:
from sdpc.io import (
    find_dynamics_checkpoint,
    find_nn_checkpoint,
    find_policy_checkpoint,
)
from sdpc.sindy import load_model
from sdpc.training import load_nn_policy

dynamics_path = find_dynamics_checkpoint(RESULTS, cfg)
policy_path = find_policy_checkpoint(RESULTS, cfg)
nn_path = find_nn_checkpoint(RESULTS, cfg)
sindy = load_model(dynamics_path, device=device)
policy = load_model(policy_path, device=device)
nn_policy = load_nn_policy(system, nn_path, cfg, device=device)
print('Dynamics :', dynamics_path)
print('SD-DPC   :', policy_path)
print('NN-DPC   :', nn_path)
print('NN loss match:', getattr(nn_policy, 'dpc_loss_reference_verified', None))

## 3. Run the comparison across seeds

Compare the configured methods.


In [ ]:
from sdpc.eval import dpc_loss_reference, evaluate

print('Canonical loss reference:', dpc_loss_reference(system, cfg))
out = evaluate(system, sindy, policy, cfg, device, nn_policy=nn_policy)
print(f"Evaluated {len(cfg.get('methods', []))} methods over {cfg['n_seeds']} seeds.")

## 4. Aggregated metric table

Compare the configured methods.


In [ ]:
from sdpc.plotting import format_metrics_table

print(format_metrics_table(out['summary']))

## 5. Per-seed detail for one metric

Compare the configured methods.


In [ ]:
import numpy as np

metric = 'tracking_mse'
for method, rows in out['per_seed'].items():
    vals = [r[metric] for r in rows if metric in r]
    print(f'{method:>24}: ' + ', '.join(f'{v:.3g}' for v in vals))

## 6. Representative controller comparison

All three trajectories use the first sampled evaluation scenario and the same exact nominal plant.

In [ ]:
import matplotlib.pyplot as plt
from sdpc.plotting import plot_controller_evaluation

fig, axes = plot_controller_evaluation(
    system, cfg, out['examples'], out['example_scenario']
)
figure_path = RESULTS / 'figures' / 'evaluation_mpc_sd_nn.png'
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)

## 7. Save results (optional)

No command-line runner is maintained for this legacy model; save directly from the notebook below.

In [ ]:
# from sdpc.io import new_run_dir, snapshot_config
# run_dir = new_run_dir(RESULTS / 'eval')
# out2 = evaluate(system, sindy, policy, cfg, device, out_dir=run_dir, nn_policy=nn_policy)
# snapshot_config(run_dir, cfg)
# print('Saved to', run_dir)